# 01 Data Exploration

EDA comparative entre PlantVillage et PlantDoc pour l'étape 4.3.

Objectifs :
- analyser la distribution des classes dans PlantVillage train/val,
- visualiser quelques exemples par classe,
- calculer des class weights pour l'entraînement,
- vérifier les dimensions des images,
- analyser la couverture réelle de PlantDoc après alignement,
- illustrer visuellement le domain gap PlantVillage vs PlantDoc.

Important : le périmètre du modèle reste celui des classes retenues dans PlantVillage. Si PlantDoc ne contient pas certaines classes, cela crée une limite de couverture du benchmark OOD, pas une raison de retirer ces classes du projet.


In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight

plt.style.use("ggplot")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

RAW_PLANTVILLAGE_DIR = REPO_ROOT / "data" / "raw" / "plantvillage"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
TEST_OOD_DIR = REPO_ROOT / "data" / "test_ood"


def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTENSIONS


def collect_species_records(species_dir: Path) -> pd.DataFrame:
    records = []
    for split_dir in sorted(path for path in species_dir.iterdir() if path.is_dir()):
        for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
            for image_path in sorted(path for path in class_dir.iterdir() if is_image_file(path)):
                records.append(
                    {
                        "task": "species",
                        "split": split_dir.name,
                        "species": class_dir.name,
                        "label": class_dir.name,
                        "image_path": str(image_path),
                    }
                )
    return pd.DataFrame(records)


def collect_disease_records(processed_dir: Path) -> pd.DataFrame:
    records = []
    for species_dir in sorted(path for path in processed_dir.iterdir() if path.is_dir()):
        if species_dir.name == "species":
            continue
        for split_dir in sorted(path for path in species_dir.iterdir() if path.is_dir()):
            for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
                for image_path in sorted(path for path in class_dir.iterdir() if is_image_file(path)):
                    records.append(
                        {
                            "task": species_dir.name,
                            "split": split_dir.name,
                            "species": species_dir.name,
                            "label": class_dir.name,
                            "image_path": str(image_path),
                        }
                    )
    return pd.DataFrame(records)


def collect_ood_records(test_ood_dir: Path) -> pd.DataFrame:
    records = []
    for species_dir in sorted(path for path in test_ood_dir.iterdir() if path.is_dir()):
        for class_dir in sorted(path for path in species_dir.iterdir() if path.is_dir()):
            for image_path in sorted(path for path in class_dir.iterdir() if is_image_file(path)):
                records.append(
                    {
                        "task": species_dir.name,
                        "split": "test_ood",
                        "species": species_dir.name,
                        "label": class_dir.name,
                        "image_path": str(image_path),
                    }
                )
    return pd.DataFrame(records)


def summarize_dimensions(image_paths) -> pd.DataFrame:
    counts = Counter()
    for image_path in image_paths:
        with Image.open(image_path) as image:
            counts[image.size] += 1
    rows = [
        {"width": width, "height": height, "count": count}
        for (width, height), count in counts.items()
    ]
    return pd.DataFrame(rows).sort_values(["count", "width", "height"], ascending=[False, True, True])


def show_examples(frame: pd.DataFrame, title: str, ncols: int = 4) -> None:
    n_images = len(frame)
    nrows = int(np.ceil(n_images / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    fig.suptitle(title, fontsize=14)

    for ax, row in zip(axes, frame.itertuples(index=False)):
        image = Image.open(row.image_path).convert("RGB")
        ax.imshow(image)
        ax.set_title(row.label, fontsize=9)
        ax.axis("off")

    for ax in axes[n_images:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


species_df = collect_species_records(PROCESSED_DIR / "species")
disease_df = collect_disease_records(PROCESSED_DIR)
ood_df = collect_ood_records(TEST_OOD_DIR)

pd.DataFrame(
    [
        {"dataset": "processed/species", "images": len(species_df)},
        {"dataset": "processed/diseases", "images": len(disease_df)},
        {"dataset": "test_ood", "images": len(ood_df)},
    ]
)


## Section A — PlantVillage (train/val)

On utilise `data/processed/` pour analyser les données d'entraînement/validation après réorganisation par tâche.


In [ ]:
species_counts = species_df.groupby(["split", "species"]).size().reset_index(name="count")

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, split in zip(axes, ["train", "val"]):
    subset = species_counts[species_counts["split"] == split].sort_values("count", ascending=False)
    ax.bar(subset["species"], subset["count"], color="#2f855a")
    ax.set_title(f"Classification d'espèce — {split}")
    ax.set_xlabel("Espèce")
    ax.set_ylabel("Nombre d'images")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

species_counts.pivot(index="species", columns="split", values="count").fillna(0).astype(int)


In [ ]:
for species in sorted(disease_df["species"].unique()):
    subset = (
        disease_df[disease_df["species"] == species]
        .groupby(["split", "label"])
        .size()
        .reset_index(name="count")
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 4), sharey=True)
    for ax, split in zip(axes, ["train", "val"]):
        split_subset = subset[subset["split"] == split].sort_values("count", ascending=True)
        ax.barh(split_subset["label"], split_subset["count"], color="#3182ce")
        ax.set_title(f"{species.title()} — {split}")
        ax.set_xlabel("Nombre d'images")
    plt.tight_layout()
    plt.show()


In [ ]:
for species in sorted(disease_df["species"].unique()):
    train_subset = disease_df[
        (disease_df["species"] == species)
        & (disease_df["split"] == "train")
    ].sort_values(["label", "image_path"])

    examples = train_subset.groupby("label", group_keys=False).head(1).reset_index(drop=True)
    show_examples(examples, title=f"{species.title()} — 1 image par classe", ncols=4)

    classes = np.unique(train_subset["label"])
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=train_subset["label"],
    )
    display(pd.DataFrame({"species": species, "label": classes, "class_weight": weights}))


In [ ]:
dimension_summary = summarize_dimensions(disease_df["image_path"])
dimension_summary

expected_dimensions = {(224, 224), (256, 256)}
observed_dimensions = set(zip(dimension_summary["width"], dimension_summary["height"]))
unexpected_dimensions = observed_dimensions - expected_dimensions

if unexpected_dimensions:
    print("Dimensions inattendues détectées :", sorted(unexpected_dimensions))
else:
    print("Dimensions conformes au guide : uniquement 224x224 ou 256x256.")


## Section B — PlantDoc (test OOD)

On analyse ici `data/test_ood/`, reconstruit après alignement des labels PlantDoc vers les labels projet.

Point méthodologique important : PlantDoc est un jeu de test OOD partiel. Il ne couvre pas toutes les classes retenues dans le projet. Les classes absentes restent conservées dans `data/processed/` et dans le périmètre du modèle ; elles ne sont simplement pas encore évaluées en OOD avec ce dataset.


In [ ]:
ood_counts = ood_df.groupby(["species", "label"]).size().reset_index(name="count")

for species in sorted(ood_df["species"].unique()):
    subset = ood_counts[ood_counts["species"] == species].sort_values("count", ascending=True)
    plt.figure(figsize=(10, 4))
    plt.barh(subset["label"], subset["count"], color="#dd6b20")
    plt.title(f"PlantDoc OOD — {species.title()}")
    plt.xlabel("Nombre d'images")
    plt.tight_layout()
    plt.show()

ood_counts[ood_counts["count"] < 50].sort_values(["count", "species", "label"])


In [ ]:
plantvillage_coverage = disease_df.groupby(["species", "label"]).size().reset_index(name="plantvillage_count")
plantdoc_coverage = ood_df.groupby(["species", "label"]).size().reset_index(name="plantdoc_count")

coverage = plantvillage_coverage.merge(
    plantdoc_coverage,
    on=["species", "label"],
    how="left",
)
coverage["plantdoc_count"] = coverage["plantdoc_count"].fillna(0).astype(int)
coverage["covered_in_ood"] = coverage["plantdoc_count"] > 0
coverage

coverage[coverage["covered_in_ood"] == False]

print(
    "Les classes absentes de PlantDoc restent conservées dans le projet : elles sont entraînées et restent testables plus tard sur un autre dataset ou sur des images personnelles."
)

comparison_targets = [
    ("tomato", "Early_Blight"),
    ("apple", "Apple_Scab"),
    ("grape", "Black_Rot"),
    ("potato", "Late_Blight"),
]

for species, label in comparison_targets:
    plantvillage_match = disease_df[
        (disease_df["species"] == species)
        & (disease_df["label"] == label)
        & (disease_df["split"] == "val")
    ].sort_values("image_path")

    if plantvillage_match.empty:
        plantvillage_match = disease_df[
            (disease_df["species"] == species)
            & (disease_df["label"] == label)
            & (disease_df["split"] == "train")
        ].sort_values("image_path")

    plantdoc_match = ood_df[
        (ood_df["species"] == species)
        & (ood_df["label"] == label)
    ].sort_values("image_path")

    if plantvillage_match.empty or plantdoc_match.empty:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f"{species.title()} — {label}", fontsize=14)

    axes[0].imshow(Image.open(plantvillage_match.iloc[0]["image_path"]).convert("RGB"))
    axes[0].set_title("PlantVillage")
    axes[0].axis("off")

    axes[1].imshow(Image.open(plantdoc_match.iloc[0]["image_path"]).convert("RGB"))
    axes[1].set_title("PlantDoc")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

print(
    "Lecture attendue : PlantVillage montre souvent un fond blanc, une feuille bien centrée et un éclairage uniforme, "
    "alors que PlantDoc introduit fond naturel, ombres, angle variable et bruit visuel."
)


## Conclusion

- PlantVillage est homogène et adapté à l'entraînement.
- Les class weights permettent de compenser le déséquilibre des classes.
- PlantDoc ne couvre pas toutes les classes projet et certaines classes sont peu représentées.
- Les classes absentes de PlantDoc ne sont pas supprimées du projet : elles restent dans le train/val et pourront être évaluées plus tard sur un autre jeu OOD ou sur des images perso.
- Le domain gap est visuellement net entre fond blanc contrôlé et images terrain.
- Ce constat justifie une évaluation séparée in-distribution vs OOD dans les étapes suivantes.
